# 🖼️ Notebook 04: CLIP Baseline: Multi-Frame Pooling, Architecture & Visual Feature Extraction
### Capstone: VL-JEPA vs. CLIP for CCTV Surveillance Event Retrieval, Event Summarization & Hallucination Analysis
---
**Objective:** Implement the static contrastive **CLIP (ViT-B/32)** video retrieval baseline:
- Temporal aggregation strategies: Mean Frame Pooling vs Temporal Attention Pooling.
- Visual encoding on the NVIDIA RTX 4000 Ada Generation GPU.
- Verification of $512$-dimensional L2-normalized visual representations for all 5,153 clips.
- Baseline limitations on surveillance video: lack of explicit temporal modeling, degradation under viewpoint variance.


## Step 1: Initialize CLIP Visual & Text Encoders
Load the pretrained CLIP model (`openai/clip-vit-base-patch32`) on CUDA FP16.


In [1]:
import sys
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import CLIPModel, CLIPTokenizer, CLIPImageProcessor
import numpy as np

sys.path.append(str(Path.cwd().parent))
import notebook_utils as nu

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch32"

print(f"[CLIP Baseline] Loading {model_id} on {device}...")
clip_model = CLIPModel.from_pretrained(model_id).to(device).eval()
clip_tokenizer = CLIPTokenizer.from_pretrained(model_id)
clip_processor = CLIPImageProcessor.from_pretrained(model_id)

print(f"[CLIP Baseline] Model successfully loaded! Embedding dimension: {clip_model.config.projection_dim}")


[CLIP Baseline] Loading openai/clip-vit-base-patch32 on cuda...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[CLIP Baseline] Model successfully loaded! Embedding dimension: 512


## Step 2: Temporal Aggregation Architectures
Contrast mean temporal pooling with temporal attention pooling across frame sequences.


In [2]:
class TemporalMeanPooling(torch.nn.Module):
    """Standard CLIP video baseline: uniform frame mean pooling."""
    def forward(self, frame_features: torch.Tensor) -> torch.Tensor:
        # frame_features: (B, T, D)
        pooled = frame_features.mean(dim=1)
        return F.normalize(pooled, p=2, dim=-1)

pooler = TemporalMeanPooling()
dummy_frames = torch.randn(4, 8, 512)
pooled_out = pooler(dummy_frames)
print(f"Input shape: {dummy_frames.shape} -> Pooled output: {pooled_out.shape} (Norm: {pooled_out.norm(dim=-1).numpy()})")


Input shape: torch.Size([4, 8, 512]) -> Pooled output: torch.Size([4, 512]) (Norm: [1.         0.99999994 1.         0.99999994])


## Step 3: Inspect Extracted CLIP Visual Embeddings
Load and verify the pre-extracted CLIP visual embeddings for the 5,153 clips across all 4 datasets.


In [3]:
embs, metadata = nu.load_all_datasets_data()

print(f"Visual Embeddings Matrix Shape: {embs.shape}")
print(f"Data type:                       {embs.dtype}")
norms = np.linalg.norm(embs, axis=-1)
print(f"L2 Norm Check (Min/Max/Mean):   {norms.min():.4f} / {norms.max():.4f} / {norms.mean():.4f}")
assert np.allclose(norms, 1.0, atol=1e-3), "Embeddings are not unit normalized!"
print("[VERIFIED] All 5,153 visual embeddings are strictly unit-normalized in R^512!")


[Dataset Loader] Loaded RLVS: 2000 clips, dim 512
[Dataset Loader] Loaded UCF_ARG: 1410 clips, dim 512
[Dataset Loader] Loaded TINYVIRAT: 743 clips, dim 512
[Dataset Loader] Loaded UCF_CRIME: 1000 clips, dim 512
Visual Embeddings Matrix Shape: (5153, 512)
Data type:                       float32
L2 Norm Check (Min/Max/Mean):   1.0000 / 1.0000 / 1.0000
[VERIFIED] All 5,153 visual embeddings are strictly unit-normalized in R^512!
